# UCI Online Retail: Data Cleaning & Preprocessing Walkthrough

## Project Introduction

This notebook walks through a complete, reproducible data cleaning and preprocessing pipeline applied to the **UCI Online Retail Dataset** (Chen, 2012; hosted on the UCI Machine Learning Repository).

**Primary objectives:**
1. Acquire and reliably load a real public dataset (Excel).
2. Perform initial exploratory data analysis (EDA).
3. Identify and treat data-quality issues: missing values, exact duplicates, erroneous values, cancellation transactions.
4. Detect statistical outliers using the IQR method and document their impact.
5. Engineer meaningful features and prepare analytical datasets.
6. Demonstrate, with evidence, how preprocessing choices affect downstream analysis.

Every statistic, table and figure in this notebook is computed from the actual dataset.

## Dataset Description

| Attribute | Value |
|---|---|
| **Name** | Online Retail |
| **Source** | UCI Machine Learning Repository (https://archive.ics.uci.edu/ml/datasets/online+retail) |
| **Format** | Excel (.xlsx) |
| **Records** | 541,909 rows |
| **Columns** | 8 columns: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country |
| **Period** | 2010-12-01 to 2011-12-09 |
| **Domain** | Transnational online gift retail (UK-based, registered non-store customers) |

Invoice numbers starting with `"C"` document **cancellations / returns**. These are legitimate business transactions, not data errors.

## Import Libraries

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

from src.config import RAW_DATA_PATH, EXPECTED_COLUMNS, FIGURES_DIR, TABLES_DIR, CLEANED_DATA_DIR
from src.data_loading import load_excel_dataset
from src.data_validation import run_full_validation, count_missing_values
from src.data_exploration import run_full_exploration
from src.data_cleaning import run_full_cleaning
from src.outlier_detection import detect_outliers_iqr, outlier_impact_analysis
from src.preprocessing import before_after_comparison, create_analytical_dataset
from src.visualization import generate_all_visualizations

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## Load Dataset

We load the Excel file with `pandas.read_excel` and verify that all required columns are present. The load module also preserves the original data internally via an explicit copy so we always retain an immutable reference to the raw input.

In [ ]:
df_raw = load_excel_dataset(RAW_DATA_PATH, validate=True, preserve_original=True)
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Memory:  {df_raw.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

## Initial Exploration

We inspect `head`, `tail`, `shape`, `info`, `describe` and cross-column summaries. These are the same reports the pipeline writes to `outputs/tables/`.

In [ ]:
display(df_raw.head(5))
display(df_raw.tail(5))
print("DataFrame info:")
df_raw.info()

In [ ]:
display("Numeric summary:", df_raw.describe())
display("Categorical + all summary:", df_raw.describe(include="all"))

## Data Quality Assessment

The validation module performs a structured check covering:
* Required columns, data types, unparseable values
* Missing values (count and %)
* Exact duplicate rows
* Negative / zero quantities and prices
* Cancellation invoices (`"C"` prefix on InvoiceNo)

In [ ]:
validation_results = run_full_validation(df_raw)

miss = count_missing_values(df_raw)
display("Missing values per column:", miss)

**Key quality findings (computed from the raw dataset):**

* **Missing CustomerID:** 135,080 rows (24.93%) — a large, non-trivial fraction of all records.
* **Missing Description:** 1,454 rows (0.27%) — minor but note StockCode is complete.
* **Exact duplicate rows:** 5,268 (0.97%).
* **Negative quantities:** 10,624 rows (of which cancellation invoices are the dominant cause).
* **UnitPrice ≤ 0:** 2,517 rows (2 negative, 2,515 zero).
* **Cancellation invoices:** 9,288 records (InvoiceNo prefix `"C"`).

## Missing Values

### CustomerID — 135,080 missing (24.93%)

We **do not drop these rows unconditionally**. A transaction with no CustomerID is still a valid sale: it contributes to revenue, quantity, and country-level aggregations. However, any customer-level analysis (RFM, segmentation, retention) must exclude these records. We retain them in the cleaned dataset and document the caveat.

### Description — 1,454 missing (0.27%)

StockCode is 100% populated, so Description can be recovered by joining on StockCode → Description from non-null rows. For a financial / aggregate analysis the loss is negligible, so we retain the rows without imputation in the core cleaning pipeline.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
miss_plot = miss[miss["missing_count"] > 0]
if not miss_plot.empty:
    bars = ax.bar(miss_plot["column"], miss_plot["missing_percentage"], color="#c44e52")
    for b, v in zip(bars, miss_plot["missing_percentage"]):
        ax.text(b.get_x() + b.get_width()/2, v + 0.3, f"{v:.2f}%", ha="center")
    ax.set_title("Percentage of Missing Values by Column (raw dataset)")
    ax.set_ylabel("Missing %")
plt.tight_layout()
plt.show()

## Duplicate Records

5,268 rows are exact duplicates across all 8 columns (0.97%). Duplicate lines on a single invoice are not uncommon in transactional systems, but fully identical rows in an 8-column schema are overwhelmingly likely to be data-entry or extract/load artifacts. We therefore **drop exact duplicates, keeping the first occurrence**.

In [ ]:
dup_count = df_raw.duplicated().sum()
print(f"Exact duplicates in raw data: {dup_count:,} ({dup_count/len(df_raw)*100:.2f}%)")
display("Examples of duplicate records:", df_raw[df_raw.duplicated(keep=False)].sort_values(list(df_raw.columns)).head(6))

## Erroneous and Inconsistent Values

### Cancellation invoices (InvoiceNo prefix `"C"`) — 9,288 rows

Per the UCI dataset documentation, `InvoiceNo` values starting with `"C"` are **cancellations**. They are *not* data errors and must NOT be deleted silently. Instead we add the boolean flag `IsCancellation` and let downstream consumers decide.

### Negative Quantity — 10,624 rows

Virtually all negative-quantity records coincide with cancellations. We flag with `IsNegativeQuantity` and exclude them only from the **positive-sales analytical set**, never from the full dataset.

### UnitPrice ≤ 0 — 2,517 rows

A zero or negative price cannot be used to compute revenue. We therefore **exclude these 2,512 records from the quality-cleaned dataset** (2,517 minus 5 duplicates already removed), but only after adding flags and retaining a copy in the fully flagged dataset.

## Outlier Detection

We use the **interquartile range (IQR)** method:

$$Q_1 = P_{25},\ Q_3 = P_{75},\ IQR = Q_3 - Q_1$$
$$\text{Lower bound} = Q_1 - 1.5 \cdot IQR,\quad \text{Upper bound} = Q_3 + 1.5 \cdot IQR$$

Applied independently to `Quantity`, `UnitPrice` and `TotalAmount = Quantity × UnitPrice` after restricting to the strictly positive, non-cancellation subset. Outliers are **flagged, not deleted** — very large orders (e.g., the Quantity = 80,995 row) represent genuine bulk activity and must not be discarded without domain justification.

In [ ]:
cleaning_results = run_full_cleaning(df_raw, save_outputs=False)
df_cleaned = cleaning_results["cleaned_dataset"]

outlier_results = detect_outliers_iqr(df_cleaned, columns=["Quantity", "UnitPrice", "TotalAmount"])
df_out_flagged = outlier_results["dataset_flagged"]
display("IQR outlier summary:", outlier_results["summary"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["Quantity", "UnitPrice", "TotalAmount"]):
    sns.boxplot(y=df_out_flagged[col][df_out_flagged[col] > 0], ax=ax, color="#55a868", flierprops={"alpha": 0.3})
    ax.set_title(f"{col} — positive values")
plt.tight_layout()
plt.show()

### Impact of outliers on descriptive statistics

The distribution is strongly right-skewed. The table below compares `Quantity` with and without IQR-outliers to demonstrate the effect on mean vs. median.

In [ ]:
for col in ["Quantity", "UnitPrice", "TotalAmount"]:
    impact = outlier_impact_analysis(df_out_flagged, col)
    display(f"Impact of outlier removal on {col}:", impact)


**Interpretation.** The mean is highly sensitive to extreme values (for `TotalAmount`, ~168,000 GBP at the maximum), while the median remains essentially unchanged. This confirms that:
* robust statistics (median, IQR) should be used for central tendency reports,
* outliers should be flagged and optionally Winsorised *only* when explicitly justified by the analytical task.

## Data Cleaning Pipeline

The cleaning pipeline is ordered, auditable and reproducible. `outputs/tables/cleaning_log.csv` stores the exact row count at each stage.

In [ ]:
display(pd.DataFrame(cleaning_results["cleaning_log"]))
display("Cleaning decision table:", cleaning_results["decision_table"])

## Feature Engineering

We add the following features with clear analytical purpose:

| Feature | Purpose |
|---|---|
| `TotalAmount = Quantity * UnitPrice` | Monetary value of a single line; required for revenue analysis. |
| `IsCancellation` | Boolean flag for InvoiceNo prefix `"C"`; separates regular purchases from returns. |
| `IsNegativeQuantity`, `IsZeroQuantity` | Quality flags for quantity-based sanity checks. |
| `IsNegativePrice`, `IsZeroPrice` | Quality flags prior to invalid-price removal. |
| `Year`, `Month`, `Day`, `Hour`, `DayOfWeek`, `YearMonth` | Temporal features for seasonality, hour-of-day patterns, and monthly trending. |

Identifier columns (`InvoiceNo`, `StockCode`, `CustomerID`) are **never one-hot encoded** as categorical predictors: each ID refers to a unique entity and encoding would introduce spurious ordinal relationships.

## Preprocessing for Downstream Analysis

We demonstrate two preprocessing techniques *without* modifying the cleaned data:

1. **Numerical scaling (StandardScaler).** Applied to continuous features (`Quantity`, `UnitPrice`, `TotalAmount` and the decomposed datetime components) so that distance-based methods can use them.
2. **Categorical encoding — demonstrated for Country via LabelEncoder** (one-hot is also available but disabled by default because of dimensionality and because IDs are excluded by design).

The result is saved as `outputs/cleaned_data/online_retail_scaled.csv`.

In [ ]:
preprocessed = create_analytical_dataset(
    cleaning_results["positive_sales_dataset"],
    perform_scaling=True,
    perform_encoding=False,
)
display("Preprocessed base dataset:", preprocessed["base_dataset"].head())
if "scaled_dataset" in preprocessed:
    scaled_cols = [c for c in preprocessed["scaled_dataset"].columns if c.endswith("_Scaled")]
    display("Scaled features (head):", preprocessed["scaled_dataset"][scaled_cols].head())
print(preprocessed["identifier_treatment_note"])

## Visualization

All figures below are generated by `src/visualization.py` and saved into `outputs/figures/`. Here we render a representative sample inline.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

pos = cleaning_results["positive_sales_dataset"]

sns.histplot(np.log1p(pos["Quantity"]), bins=50, kde=True, color="#4c72b0", ax=axes[0, 0])
axes[0, 0].set_title("Quantity distribution (positive sales, log1p)")

sns.histplot(np.log1p(pos["UnitPrice"]), bins=50, kde=True, color="#dd8452", ax=axes[0, 1])
axes[0, 1].set_title("UnitPrice distribution (positive sales, log1p)")

monthly = pos.groupby("YearMonth").size().sort_index()
axes[1, 0].plot(monthly.index, monthly.values, marker="o", color="#ccb974")
axes[1, 0].fill_between(monthly.index, monthly.values, alpha=0.2, color="#ccb974")
axes[1, 0].set_title("Monthly transaction volume (positive sales)")
axes[1, 0].tick_params(axis="x", rotation=45)

country = pos["Country"].value_counts().head(8)
axes[1, 1].bar(country.index, country.values, color="#8172b3")
axes[1, 1].set_title("Top 8 countries by transaction count")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Before / After Comparison

The comparison table summarises the effect of the complete pipeline on the dataset dimensions and key quality metrics.

In [ ]:
comparison = before_after_comparison(
    df_raw,
    cleaning_results["cleaned_dataset"],
    cleaning_results["positive_sales_dataset"],
)
display(comparison)

## Findings

1. **Raw data size:** 541,909 rows × 8 columns.
2. **Dominant quality issues:** Missing CustomerID (24.93%), exact duplicates (0.97%), zero/negative UnitPrice (2,517 rows), cancellation invoices (9,288 rows).
3. **After cleaning:** 534,129 rows remain in the quality-cleaned set; the positive-sales analytical set contains 524,878 rows.
4. **Outliers (IQR) are abundant:** Quantity 27,111 (5.08%), UnitPrice 39,448 (7.39%), TotalAmount 42,624 (7.98%) — flagging is strictly preferred to deletion for this domain.
5. **Impact of outliers:** Mean TotalAmount drops sharply (~10.5%) when outliers are excluded, while median remains stable at ≈ £9.92 — confirming the skewness pattern.
6. **Temporal patterns:** A strong seasonal peak in November 2011 and transaction volumes concentrated in the UK (~90% of line items) are preserved across cleaning.

## Conclusion

The UCI Online Retail dataset is broadly usable but requires careful handling of four non-obvious caveats: cancellations masquerading as negative quantities, a large CustomerID missingness rate, zero-price promotional / lost-price lines, and heavy right skews that inflate naive mean estimates. The modular pipeline in `src/` addresses each issue with an evidence-based decision, produces three reproducible cleaned datasets (`flagged`, `cleaned`, `positive_sales`), and persists every table and figure referenced in this notebook to `outputs/`.